In [1]:
print("hello")

hello


In [2]:
from docplex.mp.model import Model

# === Constants ===
nVehicles = 150
MinUtilTime = 400
MaxUtilTime = 500
vehicle_fixed_cost = 10000

# freq1 = [0, 0, 0, 0, 3, 3, 4, 8, 10, 10, 12, 16, 9, 15, 15, 16, 14, 12, 9, 6, 6, 4, 0, 0]
# freq2 = [0, 0, 0, 0, 4, 6 ,5, 8, 10, 10, 12, 16, 9, 15, 15, 16, 14, 12, 9, 6, 6, 4, 0, 0]
freq1 = [0, 0, 0, 0, 3, 3, 4, 8, 0, 0, 2, 16, 9, 4, 5, 5, 14, 2, 9, 6, 6, 4, 0, 0]
freq2 = [0, 0, 0, 0, 4, 6 ,5, 8, 0, 0, 2, 16, 9, 5, 5, 6, 4, 2, 9, 6, 6, 4, 0, 0]

H = range(24)
K = range(nVehicles)

# === Node generation ===
L = ["HSK", "ATB", "X"]
nodes = []
node_id = 1

for hour in range(24):
    for i in range(freq1[hour]):
        time_val = hour * 60 + (i * 60 // max(freq1[hour], 1))
        nodes.append({'id': node_id, 'time': time_val, 'loc': 'HSK'})
        node_id += 1

    for i in range(freq2[hour]):
        time_val = hour * 60 + (i * 60 // max(freq2[hour], 1))
        nodes.append({'id': node_id, 'time': time_val, 'loc': 'ATB'})
        node_id += 1


# Add depot start and end nodes
nodes.insert(0, {'id': node_id, 'time': 0, 'loc': 'X'})
id0 = node_id
node_id += 1
nodes.append({'id': node_id, 'time': 1440, 'loc': 'X'})
idend = node_id

V = nodes
nNodes = len(V)

# === Arc data ===
arc_data = {
    ("X", "X"): (0, 150),
    ("X", "HSK"): (10, 150),
    ("X", "ATB"): (40, 150),
    ("HSK", "HSK"): (200, 150),
    ("ATB", "ATB"): (200, 150),
    ("HSK", "ATB"): (110, 1),
    ("ATB", "HSK"): (110, 1),
    ("HSK", "X"): (10, 150),
    ("ATB", "X"): (40, 150)
}

# === Generate arcs E ===
E = []
for i in range(len(V)):
    for j in range(len(V)):
        if i == j:
            continue
        n1, n2 = V[i], V[j]
        if n2['time'] > n1['time']:
            loc_pair = (n1['loc'], n2['loc'])
            if loc_pair in arc_data:
                cost, cap = arc_data[loc_pair]
                E.append({"src": n1, "dst": n2, "cost": cost, "cap": cap})

# === Model ===
mdl = Model("VehicleScheduling")
mdl.context.cplex_parameters.lpmethod = 3

# Decision variables
x = mdl.binary_var_dict(((e['src']['id'], e['dst']['id'], k) for e in E for k in K), name='x')
time_vars = mdl.continuous_var_dict(((node['id'], k) for node in V for k in K), name="time")
z = mdl.binary_var_dict(K, name='z')
T = mdl.continuous_var_dict(K, name='T', lb=0)

# Arrival time variables
arrival_time = mdl.continuous_var_dict(((k, node['id']) for k in K for node in V), name='arr', lb=0, ub=1440)

# Slack variables for frequency constraints
freq_slack_ATB = mdl.continuous_var_dict(H, name='freq_slack_ATB', lb=0)
freq_slack_HSK = mdl.continuous_var_dict(H, name='freq_slack_HSK', lb=0)
freq_penalty = 10000

# Objective function
mdl.minimize(
    mdl.sum(e['cost'] * x[e['src']['id'], e['dst']['id'], k] for e in E for k in K)
    + mdl.sum(vehicle_fixed_cost * z[k] for k in K)
    + mdl.sum(freq_penalty * (freq_slack_ATB[h] + freq_slack_HSK[h]) for h in H)
)

# Fix arrival time at depot start node for each vehicle
for k in K:
    mdl.add_constraint(arrival_time[k, id0] == 0)

# Flow constraints and linking x and z
bigM = len(E)
for k in K:
    mdl.add_constraint(mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E) <= bigM * z[k])
    mdl.add_constraint(z[k] <= mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E))

# Flow conservation constraints for each node
for k in K:
    mdl.add_constraint(mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E if e['src']['id'] == id0) == z[k])
    mdl.add_constraint(mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E if e['dst']['id'] == idend) == z[k])
    for node in V:
        i = node['id']
        if i != id0 and i != idend:
            mdl.add_constraint(
                mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E if e['src']['id'] == i) ==
                mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E if e['dst']['id'] == i)
            )

# Capacity constraints for arcs with capacity 1
for e in E:
    if e['cap'] == 1:
        mdl.add_constraint(mdl.sum(x[e['src']['id'], e['dst']['id'], k] for k in K) <= 1)

# Utilization time constraints
for k in K:
    mdl.add_constraint(
        T[k] == mdl.sum(e['cost'] * x[e['src']['id'], e['dst']['id'], k] for e in E if e['cap'] == 1)
    )

    mdl.add_constraint(T[k] >= MinUtilTime * z[k])
    mdl.add_constraint(T[k] <= MaxUtilTime * z[k])

# Time feasibility constraints (arrival time with Big-M)
big_M = 1440  # 24 hours in minutes
for e in E:
    for k in K:
        i, j = e['src']['id'], e['dst']['id']
        travel_time = e['cost']

        mdl.add_constraint(
            arrival_time[k, j] >= arrival_time[k, i] + travel_time - big_M * (1 - x[i, j, k])
            # arrival_time[k, j] >= arrival_time[k, i] + travel_time
        )

# Frequency constraints with slack
for h in H:
    mdl.add_constraint(
        mdl.sum(
            x[e['src']['id'], e['dst']['id'], k]
            for k in K
            for e in E if e['cap'] == 1 and e['src']['loc'] == 'ATB' and (e['src']['time'] // 60) == h
        ) + freq_slack_ATB[h] >= freq1[h]
    )
    mdl.add_constraint(
        mdl.sum(
            x[e['src']['id'], e['dst']['id'], k]
            for k in K
            for e in E if e['cap'] == 1 and e['src']['loc'] == 'HSK' and (e['src']['time'] // 60) == h
        ) + freq_slack_HSK[h] >= freq2[h]
    )

# Solver parameters
mdl.context.cplex_parameters.mip.tolerances.mipgap = 0.01
mdl.context.cplex_parameters.timelimit = 600
mdl.context.cplex_parameters.threads = 16
mdl.context.cplex_parameters.mip.strategy.heuristicfreq = 10

# Solve
solution = mdl.solve(log_output=True)

if solution:
    used_vehicles = [k for k in K if z[k].solution_value > 0.5]
    print(f"Number of vehicles used: {len(used_vehicles)}\n")

    for k in used_vehicles:
        print(f"Schedule for Vehicle {k + 1}: Utilization time = {T[k].solution_value:.2f}")
        current_node = id0

        while current_node != idend:
            next_arcs = [e for e in E if e['src']['id'] == current_node and x[e['src']['id'], e['dst']['id'], k].solution_value > 0.5]
            if not next_arcs:
                print("  ERROR: Route incomplete or disconnected.")
                break

            arc = next_arcs[0]
            src = arc['src']
            dst = arc['dst']

            src_h, src_m = divmod(src['time'], 60)
            dst_h, dst_m = divmod(dst['time'], 60)
            print(f"  ({src['loc']} at {src_h:02d}:{src_m:02d}) --> ({dst['loc']} at {dst_h:02d}:{dst_m:02d}), cost={arc['cost']}")

            current_node = dst['id']
        print()
else:
    print("No feasible solution found.")


KeyboardInterrupt: 

In [ ]:
from docplex.mp.model import Model

# === Constants ===
nVehicles = 150
MinUtilTime = 400
MaxUtilTime = 1500
vehicle_fixed_cost = 10000

# freq1 = [0, 0, 0, 0, 3, 3, 4, 8, 10, 10, 12, 16, 9, 15, 15, 16, 14, 12, 9, 6, 6, 4, 0, 0]
# freq2 = [0, 0, 0, 0, 4, 6 ,5, 8, 10, 10, 12, 16, 9, 15, 15, 16, 14, 12, 9, 6, 6, 4, 0, 0]
# freq1 = [0, 0, 0, 0, 3, 3, 4, 8, 0, 0, 2, 16, 9, 4, 5, 5, 14, 2, 9, 6, 6, 4, 0, 0]
# freq2 = [0, 0, 0, 0, 4, 6 ,5, 8, 0, 0, 2, 16, 9, 5, 5, 6, 4, 2, 9, 6, 6, 4, 0, 0]
freq1 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 5, 6, 2, 0, 0]
freq2 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 5, 6, 4, 0, 0]

H = range(24)
K = range(nVehicles)

# === Node generation ===
L = ["HSK", "ATB", "X"]
nodes = []
node_id = 1

for hour in range(24):
    for i in range(freq1[hour]):
        time_val = hour * 60 + (i * 60 // max(freq1[hour], 1))
        nodes.append({'id': node_id, 'time': time_val, 'loc': 'HSK'})
        node_id += 1

    for i in range(freq2[hour]):
        time_val = hour * 60 + (i * 60 // max(freq2[hour], 1))
        nodes.append({'id': node_id, 'time': time_val, 'loc': 'ATB'})
        node_id += 1


# Add depot start and end nodes
nodes.insert(0, {'id': node_id, 'time': 0, 'loc': 'X'})
id0 = node_id
node_id += 1
nodes.append({'id': node_id, 'time': 1440, 'loc': 'X'})
idend = node_id

V = nodes
nNodes = len(V)

# === Arc data ===
arc_data = {
    ("X", "X"): (0, 150),
    ("X", "HSK"): (10, 150),
    ("X", "ATB"): (40, 150),
    ("HSK", "HSK"): (200, 150),
    ("ATB", "ATB"): (200, 150),
    ("HSK", "ATB"): (110, 1),
    ("ATB", "HSK"): (110, 1),
    ("HSK", "X"): (10, 150),
    ("ATB", "X"): (40, 150)
}

# === Generate arcs E ===
E = []
for i in range(len(V)):
    for j in range(len(V)):
        if i == j:
            continue
        n1, n2 = V[i], V[j]
        if n2['time'] > n1['time']:
            loc_pair = (n1['loc'], n2['loc'])
            if loc_pair in arc_data:
                cost, cap = arc_data[loc_pair]
                E.append({"src": n1, "dst": n2, "cost": cost, "cap": cap})

# === Model ===
mdl = Model("VehicleScheduling")
mdl.context.cplex_parameters.lpmethod = 3

# Decision variables
x = mdl.binary_var_dict(((e['src']['id'], e['dst']['id'], k) for e in E for k in K), name='x')
z = mdl.binary_var_dict(K, name='z')
T = mdl.continuous_var_dict(K, name='T', lb=0)

# Arrival time variables
arrival_time = mdl.continuous_var_dict(((k, node['id']) for k in K for node in V), name='arr', lb=0, ub=1440)

# Slack variables for frequency constraints
freq_slack_ATB = mdl.continuous_var_dict(H, name='freq_slack_ATB', lb=0)
freq_slack_HSK = mdl.continuous_var_dict(H, name='freq_slack_HSK', lb=0)
freq_penalty = 10000

# Objective function
mdl.minimize(
    mdl.sum(e['cost'] * x[e['src']['id'], e['dst']['id'], k] for e in E for k in K)
    + mdl.sum(vehicle_fixed_cost * z[k] for k in K)
    # + mdl.sum(freq_penalty * (freq_slack_ATB[h] + freq_slack_HSK[h]) for h in H)
)

# # Fix arrival time at depot start node for each vehicle
# for k in K:
#     mdl.add_constraint(arrival_time[k, id0] == 0)

# Flow constraints and linking x and z
bigM = len(E)
for k in K:
    mdl.add_constraint(mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E) <= bigM * z[k])
    mdl.add_constraint(z[k] <= mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E))

# Flow conservation constraints for each node
for k in K:
    mdl.add_constraint(mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E if e['src']['id'] == id0) == z[k])
    mdl.add_constraint(mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E if e['dst']['id'] == idend) == z[k])
    for node in V:
        i = node['id']
        if i != id0 and i != idend:
            mdl.add_constraint(
                mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E if e['src']['id'] == i) ==
                mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E if e['dst']['id'] == i)
            )

# Capacity constraints for arcs with capacity 1
for e in E:
    if e['cap'] == 1:
        mdl.add_constraint(mdl.sum(x[e['src']['id'], e['dst']['id'], k] for k in K) <= 1)

# Utilization time constraints
for k in K:
    mdl.add_constraint(
        T[k] == mdl.sum(e['cost'] * x[e['src']['id'], e['dst']['id'], k] for e in E if e['cap'] == 1)
    )

    mdl.add_constraint(T[k] >= MinUtilTime * z[k])
    mdl.add_constraint(T[k] <= MaxUtilTime * z[k])

# Time feasibility constraints (arrival time with Big-M)
# big_M = 1440  # 24 hours in minutes
big_M = max(e['cost'] for e in E) + 30 
for e in E:
    for k in K:
        i, j = e['src']['id'], e['dst']['id']
        travel_time = e['cost']

        mdl.add_constraint(
            arrival_time[k, j] >= arrival_time[k, i] + travel_time - big_M * (1 - x[i, j, k])
            # arrival_time[k, j] >= arrival_time[k, i] + travel_time
        )

# # Frequency constraints with slack
# for h in H:
#     mdl.add_constraint(
#         mdl.sum(
#             x[e['src']['id'], e['dst']['id'], k]
#             for k in K
#             for e in E if e['cap'] == 1 and e['src']['loc'] == 'ATB' and (e['src']['time'] // 60) == h
#         ) + freq_slack_ATB[h] >= freq1[h]
#     )
#     mdl.add_constraint(
#         mdl.sum(
#             x[e['src']['id'], e['dst']['id'], k]
#             for k in K
#             for e in E if e['cap'] == 1 and e['src']['loc'] == 'HSK' and (e['src']['time'] // 60) == h
#         ) + freq_slack_HSK[h] >= freq2[h]
#     )

# Solver parameters
mdl.context.cplex_parameters.mip.tolerances.mipgap = 0.05
mdl.context.cplex_parameters.timelimit = 600
mdl.context.cplex_parameters.threads = 16
mdl.context.cplex_parameters.mip.strategy.heuristicfreq = 10

# Solve
solution = mdl.solve(log_output=True)

if solution:
    used_vehicles = [k for k in K if z[k].solution_value > 0.5]
    print(f"Number of vehicles used: {len(used_vehicles)}\n")

    for k in used_vehicles:
        print(f"Schedule for Vehicle {k + 1}: Utilization time = {T[k].solution_value:.2f}")
        current_node = id0

        while current_node != idend:
            next_arcs = [e for e in E if e['src']['id'] == current_node and x[e['src']['id'], e['dst']['id'], k].solution_value > 0.5]
            if not next_arcs:
                print("  ERROR: Route incomplete or disconnected.")
                break

            arc = next_arcs[0]
            src = arc['src']
            dst = arc['dst']

            src_h, src_m = divmod(src['time'], 60)
            dst_h, dst_m = divmod(dst['time'], 60)
            print(f"  ({src['loc']} at {src_h:02d}:{src_m:02d}) --> ({dst['loc']} at {dst_h:02d}:{dst_m:02d}), cost={arc['cost']}")

            current_node = dst['id']
        print()
else:
    print("No feasible solution found.")


In [ ]:
from docplex.mp.model import Model

# === Constants ===
nVehicles = 150
MinUtilTime = 400
MaxUtilTime = 1500
vehicle_fixed_cost = 10000

freq1 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 5, 6, 2, 0, 0]
freq2 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 5, 6, 4, 0, 0]

H = range(24)
K = range(nVehicles)

# === Node generation ===
L = ["HSK", "ATB", "X"]
nodes = []
node_id = 1

for hour in range(24):
    for i in range(freq1[hour]):
        time_val = hour * 60 + (i * 60 // max(freq1[hour], 1))
        nodes.append({'id': node_id, 'time': time_val, 'loc': 'HSK'})
        node_id += 1

    for i in range(freq2[hour]):
        time_val = hour * 60 + (i * 60 // max(freq2[hour], 1))
        nodes.append({'id': node_id, 'time': time_val, 'loc': 'ATB'})
        node_id += 1


# Add depot start and end nodes
nodes.insert(0, {'id': node_id, 'time': 0, 'loc': 'X'})
id0 = node_id
node_id += 1
nodes.append({'id': node_id, 'time': 1440, 'loc': 'X'})
idend = node_id

V = nodes
nNodes = len(V)

# === Arc data ===
arc_data = {
    ("X", "X"): (0, 150),
    ("X", "HSK"): (10, 150),
    ("X", "ATB"): (40, 150),
    ("HSK", "HSK"): (200, 150),
    ("ATB", "ATB"): (200, 150),
    ("HSK", "ATB"): (110, 1),
    ("ATB", "HSK"): (110, 1),
    ("HSK", "X"): (10, 150),
    ("ATB", "X"): (40, 150)
}

# === Generate arcs E ===
E = []
for i in range(len(V)):
    for j in range(len(V)):
        if i == j:
            continue
        n1, n2 = V[i], V[j]
        if n2['time'] > n1['time']:
            loc_pair = (n1['loc'], n2['loc'])
            if loc_pair in arc_data:
                cost, cap = arc_data[loc_pair]
                E.append({"src": n1, "dst": n2, "cost": cost, "cap": cap})

# === Model ===
mdl = Model("VehicleScheduling")
mdl.context.cplex_parameters.lpmethod = 3

# Decision variables
x = mdl.binary_var_dict(((e['src']['id'], e['dst']['id'], k) for e in E for k in K), name='x')
# time_vars = mdl.continuous_var_dict(((node['id'], k) for node in V for k in K), name="time")
z = mdl.binary_var_dict(K, name='z')
T = mdl.continuous_var_dict(K, name='T', lb=0)

# Arrival time variables
arrival_time = mdl.continuous_var_dict(((k, node['id']) for k in K for node in V), name='arr', lb=0, ub=1440)

for k in K:
    mdl.add_constraint(arrival_time[k, id0] == 0)

# Slack variables for frequency constraints
freq_slack_ATB = mdl.continuous_var_dict(H, name='freq_slack_ATB', lb=0)
freq_slack_HSK = mdl.continuous_var_dict(H, name='freq_slack_HSK', lb=0)
freq_penalty = 10000

# Objective
mdl.minimize(
    mdl.sum(e['cost'] * x[e['src']['id'], e['dst']['id'], k] for e in E for k in K)
    + mdl.sum(vehicle_fixed_cost * z[k] for k in K)
    + mdl.sum(freq_penalty * (freq_slack_ATB[h] + freq_slack_HSK[h]) for h in H)
)

# Link x and z
bigM = len(E)
for k in K:
    mdl.add_constraint(mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E) <= bigM * z[k])
    mdl.add_constraint(z[k] <= mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E))

# Flow constraints
for k in K:
    mdl.add_constraint(mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E if e['src']['id'] == id0) == z[k])
    mdl.add_constraint(mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E if e['dst']['id'] == idend) == z[k])
    for node in V:
        i = node['id']
        if i != id0 and i != idend:
            mdl.add_constraint(
                mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E if e['src']['id'] == i) ==
                mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E if e['dst']['id'] == i)
            )

# Arc capacity constraints
for e in E:
    if e['cap'] == 1:
        mdl.add_constraint(mdl.sum(x[e['src']['id'], e['dst']['id'], k] for k in K) <= 1)

# Utilization time calculation
for k in K:
    mdl.add_constraint(T[k] == mdl.sum(e['cost'] * x[e['src']['id'], e['dst']['id'], k] for e in E if e['cap'] == 1))

# Minimum utilization time per used vehicle
for k in K:
    mdl.add_constraint(T[k] >= MinUtilTime * z[k])
    mdl.add_constraint(T[k] <= MaxUtilTime * z[k])

# Time feasibility constraints (arrival time with Big-M)
big_M = 1440  # 24 hours in minutes
for e in E:
    for k in K:
        i, j = e['src']['id'], e['dst']['id']
        travel_time = e['cost']

        mdl.add_constraint(
            arrival_time[k, j] >= arrival_time[k, i] + travel_time - big_M * (1 - x[i, j, k]))

# Frequency constraints (corrected double loop)
for h in H:
    mdl.add_constraint(
        mdl.sum(
            x[e['src']['id'], e['dst']['id'], k]
            for k in K
            for e in E if e['cap'] == 1 and e['src']['loc'] == 'ATB' and (e['src']['time'] // 60) == h
        )
        + freq_slack_ATB[h] >= freq1[h]
    )
    mdl.add_constraint(
        mdl.sum(
            x[e['src']['id'], e['dst']['id'], k]
            for k in K
            for e in E if e['cap'] == 1 and e['src']['loc'] == 'HSK' and (e['src']['time'] // 60) == h
        )
        + freq_slack_HSK[h] >= freq2[h]
    )

# Solver tuning
mdl.context.cplex_parameters.mip.tolerances.mipgap = 0.01
mdl.context.cplex_parameters.timelimit = 600
mdl.context.cplex_parameters.threads = 16
mdl.context.cplex_parameters.mip.strategy.heuristicfreq = 10

# Solve
solution = mdl.solve(log_output=True)

if solution:
    used_vehicles = [k for k in K if z[k].solution_value > 0.5]
    print(f"Number of vehicles used: {len(used_vehicles)}\n")

    for k in used_vehicles:
        print(f"Schedule for Vehicle {k + 1}: Utilization time = {T[k].solution_value:.2f}")

        current_node = id0

        while current_node != idend:
            next_arcs = [e for e in E if e['src']['id'] == current_node and x[e['src']['id'], e['dst']['id'], k].solution_value > 0.5]
            if not next_arcs:
                print("  ERROR: Route incomplete or disconnected.")
                break
            
            arc = next_arcs[0]
            src = arc['src']
            dst = arc['dst']

            src_h, src_m = divmod(src['time'], 60)
            dst_h, dst_m = divmod(dst['time'], 60)

            # Adjust depot times based on travel cost
            if src['loc'] == 'X':
                src_h, src_m = divmod(int(dst['time']-arc['cost']), 60)
            
            if dst['loc'] == 'X':
                dst_h, dst_m = divmod(int(src['time']+arc['cost']), 60)
            
            print(f"  ({src['loc']} at {src_h:02d}:{src_m:02d}) --> ({dst['loc']} at {dst_h:02d}:{dst_m:02d}), cost={arc['cost']}")

            current_node = dst['id']
        print()
else:
    print("No feasible solution found.")


In [6]:
from docplex.mp.model import Model

# === Constants ===
nVehicles = 150
MinUtilTime = 400
MaxUtilTime = 1000
vehicle_fixed_cost = 10000

freq1 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 5, 6, 2, 0, 0]  # ATB
freq2 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 5, 6, 4, 0, 0]  # HSK

H = range(24)
K = range(nVehicles)

# === Node Generation ===
nodes = []
node_id = 1

for hour in range(24):
    for i in range(freq1[hour]):
        time_val = hour * 60 + (i * 60 // max(freq1[hour], 1))
        nodes.append({'id': node_id, 'time': time_val, 'loc': 'ATB'})
        node_id += 1
    for i in range(freq2[hour]):
        time_val = hour * 60 + (i * 60 // max(freq2[hour], 1))
        nodes.append({'id': node_id, 'time': time_val, 'loc': 'HSK'})
        node_id += 1

# Add start and end depot
id0 = node_id
nodes.insert(0, {'id': id0, 'time': 0, 'loc': 'X'})
node_id += 1
idend = node_id
nodes.append({'id': idend, 'time': 1440, 'loc': 'X'})

V = nodes
nNodes = len(V)

# === Arc Generation ===
arc_data = {
    ("X", "X"): (0, 150),
    ("X", "HSK"): (10, 150),
    ("X", "ATB"): (40, 150),
    ("HSK", "HSK"): (200, 150),
    ("ATB", "ATB"): (200, 150),
    ("HSK", "ATB"): (110, 1),
    ("ATB", "HSK"): (110, 1),
    ("HSK", "X"): (10, 150),
    ("ATB", "X"): (40, 150)
}

E = []
for i in range(len(V)):
    for j in range(len(V)):
        if i == j:
            continue
        n1, n2 = V[i], V[j]
        if n2['time'] > n1['time']:
            key = (n1['loc'], n2['loc'])
            if key in arc_data:
                cost, cap = arc_data[key]
                E.append({"src": n1, "dst": n2, "cost": cost, "cap": cap})

# === Model Setup ===
mdl = Model("VehicleScheduling")
mdl.context.cplex_parameters.lpmethod = 3

# Variables
x = mdl.binary_var_dict(((e['src']['id'], e['dst']['id'], k) for e in E for k in K), name='x')
z = mdl.binary_var_dict(K, name='z')
T = mdl.continuous_var_dict(K, name='T', lb=0)
arrival_time = mdl.continuous_var_dict(((k, node['id']) for k in K for node in V), name='arr', lb=0, ub=1440)

# Slack for unmet frequency
freq_slack_ATB = mdl.continuous_var_dict(H, name='slack_ATB', lb=0)
freq_slack_HSK = mdl.continuous_var_dict(H, name='slack_HSK', lb=0)

# Initial arrival time
for k in K:
    mdl.add_constraint(arrival_time[k, id0] == 0)

# Objective
freq_penalty = 10000
mdl.minimize(
    mdl.sum(e['cost'] * x[e['src']['id'], e['dst']['id'], k] for e in E for k in K) +
    mdl.sum(vehicle_fixed_cost * z[k] for k in K) +
    mdl.sum(freq_penalty * (freq_slack_ATB[h] + freq_slack_HSK[h]) for h in H)
)

# x and z linking
bigM = len(E)
for k in K:
    mdl.add_constraint(mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E) <= bigM * z[k])
    mdl.add_constraint(z[k] <= mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E))

# Flow constraints
for k in K:
    mdl.add_constraint(mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E if e['src']['id'] == id0) == z[k])
    mdl.add_constraint(mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E if e['dst']['id'] == idend) == z[k])
    for node in V:
        nid = node['id']
        if nid != id0 and nid != idend:
            mdl.add_constraint(
                mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E if e['src']['id'] == nid) ==
                mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E if e['dst']['id'] == nid)
            )

# Arc capacities
for e in E:
    if e['cap'] == 1:
        mdl.add_constraint(mdl.sum(x[e['src']['id'], e['dst']['id'], k] for k in K) <= 1)

# Utilization constraints
for k in K:
    mdl.add_constraint(T[k] == mdl.sum(
        e['cost'] * x[e['src']['id'], e['dst']['id'], k]
        for e in E if e['cap'] == 1
    ))
    mdl.add_constraint(T[k] >= MinUtilTime * z[k])
    mdl.add_constraint(T[k] <= MaxUtilTime * z[k])

# Time propagation
big_M = 1440
for e in E:
    i, j = e['src']['id'], e['dst']['id']
    travel = e['cost']
    for k in K:
        mdl.add_constraint(arrival_time[k, j] >= arrival_time[k, i] + travel - big_M * (1 - x[i, j, k]))

# Frequency constraints
for h in H:
    # ATB departures in hour h
    mdl.add_constraint(
        mdl.sum(x[e['src']['id'], e['dst']['id'], k]
                for k in K
                for e in E if e['cap'] == 1 and e['src']['loc'] == 'ATB' and (e['src']['time'] // 60) == h)
        + freq_slack_ATB[h] >= freq1[h]
    )
    # HSK departures in hour h
    mdl.add_constraint(
        mdl.sum(x[e['src']['id'], e['dst']['id'], k]
                for k in K
                for e in E if e['cap'] == 1 and e['src']['loc'] == 'HSK' and (e['src']['time'] // 60) == h)
        + freq_slack_HSK[h] >= freq2[h]
    )

# === Solve ===
mdl.context.cplex_parameters.mip.tolerances.mipgap = 0.01
mdl.context.cplex_parameters.timelimit = 600
mdl.context.cplex_parameters.threads = 16
mdl.context.cplex_parameters.mip.strategy.heuristicfreq = 10

solution = mdl.solve(log_output=True)

#output
if solution:
    used_vehicles = [k for k in K if z[k].solution_value > 0.5]
    print(f"Number of vehicles used: {len(used_vehicles)}\n")

    for k in used_vehicles:
        print(f"Schedule for Vehicle {k + 1}: Utilization time = {T[k].solution_value:.2f}")

        current_node = id0  # Ensure id0 is defined appropriately

        while current_node != idend:
            next_arcs = [e for e in E if e['src']['id'] == current_node and x[e['src']['id'], e['dst']['id'], k].solution_value > 0.5]
            if not next_arcs:
                print("  ERROR: Route incomplete or disconnected.")
                break

            arc = next_arcs[0]
            src = arc['src']
            dst = arc['dst']

            # Optionally adjust times only if necessary
            if src['loc'] == 'X':
                src_time = int(dst['time'] - arc['cost'])
            else:
                src_time = int(src['time'])

            if dst['loc'] == 'X':
                dst_time = int(src['time'] + arc['cost'])
            else:
                dst_time = int(dst['time'])

            src_h, src_m = divmod(src_time, 60)
            dst_h, dst_m = divmod(dst_time, 60)

            print(f"  ({src['loc']} at {src_h:02d}:{src_m:02d}) --> ({dst['loc']} at {dst_h:02d}:{dst_m:02d}), cost={arc['cost']}")

            current_node = dst['id']
        print()
else:
    print("No feasible solution found.")

Version identifier: 22.1.1.0 | 2022-11-27 | 9160aff4d
CPXPARAM_Read_DataCheck                          1
CPXPARAM_LPMethod                                3
CPXPARAM_Threads                                 16
CPXPARAM_MIP_Strategy_HeuristicFreq              10
CPXPARAM_TimeLimit                               600
CPXPARAM_MIP_Tolerances_MIPGap                   0.01
Found incumbent of value 420000.000000 after 0.03 sec. (16.37 ticks)
Tried aggregator 3 times.
MIP Presolve eliminated 3938 rows and 488 columns.
MIP Presolve modified 91500 coefficients.
Aggregator did 750 substitutions.
Reduced MIP has 142180 rows, 144610 columns, and 976510 nonzeros.
Reduced MIP has 138600 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 1.44 sec. (1541.94 ticks)
Probing fixed 3600 vars, tightened 0 bounds.
Probing time = 2.12 sec. (384.05 ticks)
Tried aggregator 1 time.
Detecting symmetries...
MIP Presolve eliminated 4356 rows and 3600 columns.
MIP Presolve modified 74550 coefficients.
Redu

In [30]:
from docplex.mp.model import Model

# === Constants ===
nVehicles = 150
MinUtilTime = 400
MaxUtilTime = 1000
vehicle_fixed_cost = 10000

freq1 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 5, 6, 2, 0, 0]  # ATB
freq2 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 5, 6, 4, 0, 0]  # HSK

# === Node Generation ===
nodes = []
node_id = 1

# Add distinct start and end depot
id0 = node_id
nodes.append({'id': id0, 'time': 0, 'loc': 'X'})
node_id += 1

for hour in range(24):
    for i in range(freq1[hour]):
        time_val = hour * 60 + (i * 60 // freq1[hour])
        nodes.append({'id': node_id, 'time': time_val, 'loc': 'ATB'})
        node_id += 1

    for i in range(freq2[hour]):
        time_val = hour * 60 + (i * 60 // freq2[hour])
        nodes.append({'id': node_id, 'time': time_val, 'loc': 'HSK'})
        node_id += 1

idend = node_id
nodes.append({'id': idend, 'time': 1440, 'loc': 'X'})
node_id += 1

nodes.sort(key=lambda x: x['time'])

V = nodes
nNodes = len(V)

# === Arc Generation ===
arc_data = {
    ("X", "X"): (0, 150),
    ("X", "HSK"): (10, 150),
    ("X", "ATB"): (40, 150),
    ("HSK", "HSK"): (200, 150),
    ("ATB", "ATB"): (200, 150),
    ("HSK", "ATB"): (110, 1),
    ("ATB", "HSK"): (110, 1),
    ("HSK", "X"): (10, 150),
    ("ATB", "X"): (40, 150)
}

E = []
synthetic_id = len(nodes)  # Continue IDs after real nodes

for node in V:
    src = {'id': node['id'], 'time': node['time'], 'loc': node['loc']}
    
    for (from_loc, to_loc), (cost, cap) in arc_data.items():
        if from_loc == node['loc'] and to_loc != node['loc']:  # Skip self-location arcs
            dst_time = node['time'] + cost
            if dst_time <= 1440:
                dst = {'id': synthetic_id, 'time': dst_time, 'loc': to_loc}
                E.append({
                    'src': src,
                    'dst': dst,
                    'cost': cost,
                    'cap': cap
                })
                synthetic_id += 1

                

print("Start depot node ID:", id0)
print("End depot node ID:", idend)
print("Total nodes:", len(nodes))
print("Total arcs:", len(E))

print(E)

Start depot node ID: 1
End depot node ID: 44
Total nodes: 44
Total arcs: 86
[{'src': {'id': 1, 'time': 0, 'loc': 'X'}, 'dst': {'id': 44, 'time': 10, 'loc': 'HSK'}, 'cost': 10, 'cap': 150}, {'src': {'id': 1, 'time': 0, 'loc': 'X'}, 'dst': {'id': 45, 'time': 40, 'loc': 'ATB'}, 'cost': 40, 'cap': 150}, {'src': {'id': 2, 'time': 1020, 'loc': 'ATB'}, 'dst': {'id': 46, 'time': 1130, 'loc': 'HSK'}, 'cost': 110, 'cap': 1}, {'src': {'id': 2, 'time': 1020, 'loc': 'ATB'}, 'dst': {'id': 47, 'time': 1060, 'loc': 'X'}, 'cost': 40, 'cap': 150}, {'src': {'id': 5, 'time': 1020, 'loc': 'HSK'}, 'dst': {'id': 48, 'time': 1130, 'loc': 'ATB'}, 'cost': 110, 'cap': 1}, {'src': {'id': 5, 'time': 1020, 'loc': 'HSK'}, 'dst': {'id': 49, 'time': 1030, 'loc': 'X'}, 'cost': 10, 'cap': 150}, {'src': {'id': 3, 'time': 1040, 'loc': 'ATB'}, 'dst': {'id': 50, 'time': 1150, 'loc': 'HSK'}, 'cost': 110, 'cap': 1}, {'src': {'id': 3, 'time': 1040, 'loc': 'ATB'}, 'dst': {'id': 51, 'time': 1080, 'loc': 'X'}, 'cost': 40, 'cap': 

In [55]:
from datetime import timedelta

freq1 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 5, 6, 2, 0, 0]  # ATB
freq2 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 5, 6, 4, 0, 0]  # HSK

nodes = []
node_id = 1

# Optional: Add a start node at 00:00
nodes.append({'id': node_id, 'time': 0, 'loc': 'X'})
node_id += 1

for hour in range(24):
    f1 = freq1[hour]
    f2 = freq2[hour]

    for i in range(f1):
        time_val = hour * 60 + (i * 60 // f1)
        nodes.append({'id': node_id, 'time': time_val, 'loc': 'ATB'})
        node_id += 1

    for i in range(f2):
        time_val = hour * 60 + (i * 60 // f2)
        nodes.append({'id': node_id, 'time': time_val, 'loc': 'HSK'})
        node_id += 1

# Optional: Add an end node at 24:00
nodes.append({'id': node_id, 'time': 1440, 'loc': 'X'})
node_id += 1

# for node in nodes:
#     node['hhmm'] = str(timedelta(minutes=node['time']))[:-3]

nodes.sort(key=lambda x: x['time'])

count=0
# Output preview
for node in nodes:
    # if(node['loc']=='HSK'):
        print(node)
        count+=1
print(count)














MinUtilTime = 400

# Your travel time between locations from arc_data (in minutes)
arc_data = {
    ("X", "X"): (0, 150),
    ("X", "HSK"): (10, 150),
    ("X", "ATB"): (40, 150),
    ("HSK", "HSK"): (200, 150),
    ("ATB", "ATB"): (200, 150),
    ("HSK", "ATB"): (110, 1),
    ("ATB", "HSK"): (110, 1),
    ("HSK", "X"): (10, 150),
    ("ATB", "X"): (40, 150)
}

# Allowed alternation
transitions = {
    "ATB": "HSK",
    "HSK": "ATB"
}

travel_time = {
    ("ATB", "HSK"): arc_data[("ATB", "HSK")][0],
    ("HSK", "ATB"): arc_data[("HSK", "ATB")][0]
}

synthetic_id = len(nodes) + 1
E = []

for node in nodes:
    loc = node['loc']
    time = node['time']
    src_id = node['id']

    if loc not in transitions:
        continue  # skip if not ATB or HSK

    total_time = 0
    curr_loc = loc
    curr_time = time
    curr_src_id = src_id

    while total_time < MinUtilTime:
        next_loc = transitions[curr_loc]
        cost = travel_time[(curr_loc, next_loc)]
        next_time = curr_time + cost

        if next_time > 1440:  # don't exceed the day
            break

        # Create synthetic destination node
        dst = {'id': synthetic_id, 'time': next_time, 'loc': next_loc}
        synthetic_id += 1

        # Add arc
        E.append({
            'src': {'id': curr_src_id, 'time': curr_time, 'loc': curr_loc},
            'dst': dst,
            'cost': cost,
            'cap': 1
        })

        # Update for next iteration
        total_time += cost
        curr_loc = next_loc
        curr_time = next_time
        curr_src_id = dst['id']

# Print example arcs count and some arcs:
print(f"Total generated arcs: {len(E)}")
for arc in E[:10]:
    print(f"From node {arc['src']['id']} ({arc['src']['loc']} @ {arc['src']['time']}) "
          f"to node {arc['dst']['id']} ({arc['dst']['loc']} @ {arc['dst']['time']}) "
          f"cost: {arc['cost']}")


{'id': 1, 'time': 0, 'loc': 'X'}
{'id': 2, 'time': 1020, 'loc': 'ATB'}
{'id': 5, 'time': 1020, 'loc': 'HSK'}
{'id': 3, 'time': 1040, 'loc': 'ATB'}
{'id': 6, 'time': 1040, 'loc': 'HSK'}
{'id': 4, 'time': 1060, 'loc': 'ATB'}
{'id': 7, 'time': 1060, 'loc': 'HSK'}
{'id': 8, 'time': 1080, 'loc': 'ATB'}
{'id': 12, 'time': 1080, 'loc': 'HSK'}
{'id': 9, 'time': 1095, 'loc': 'ATB'}
{'id': 13, 'time': 1095, 'loc': 'HSK'}
{'id': 10, 'time': 1110, 'loc': 'ATB'}
{'id': 14, 'time': 1110, 'loc': 'HSK'}
{'id': 11, 'time': 1125, 'loc': 'ATB'}
{'id': 15, 'time': 1125, 'loc': 'HSK'}
{'id': 16, 'time': 1140, 'loc': 'ATB'}
{'id': 21, 'time': 1140, 'loc': 'HSK'}
{'id': 17, 'time': 1152, 'loc': 'ATB'}
{'id': 22, 'time': 1152, 'loc': 'HSK'}
{'id': 18, 'time': 1164, 'loc': 'ATB'}
{'id': 23, 'time': 1164, 'loc': 'HSK'}
{'id': 19, 'time': 1176, 'loc': 'ATB'}
{'id': 24, 'time': 1176, 'loc': 'HSK'}
{'id': 20, 'time': 1188, 'loc': 'ATB'}
{'id': 25, 'time': 1188, 'loc': 'HSK'}
{'id': 26, 'time': 1200, 'loc': 'ATB'}


In [42]:
from datetime import timedelta

# Frequencies of nodes per hour at ATB and HSK
freq1 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 5, 6, 2, 0, 0]  # ATB
freq2 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 5, 6, 4, 0, 0]  # HSK

# Generate nodes with ids, times, and locations
nodes = []
node_id = 1

# Add start node at 00:00 (location 'X')
nodes.append({'id': node_id, 'time': 0, 'loc': 'X'})
node_id += 1

for hour in range(24):
    f1 = freq1[hour]
    f2 = freq2[hour]

    # Generate ATB nodes for this hour
    for i in range(f1):
        time_val = hour * 60 + (i * 60 // max(f1, 1))  # avoid div by zero
        nodes.append({'id': node_id, 'time': time_val, 'loc': 'ATB'})
        node_id += 1

    # Generate HSK nodes for this hour
    for i in range(f2):
        time_val = hour * 60 + (i * 60 // max(f2, 1))
        nodes.append({'id': node_id, 'time': time_val, 'loc': 'HSK'})
        node_id += 1

# Add end node at 24:00 (location 'X')
nodes.append({'id': node_id, 'time': 1440, 'loc': 'X'})
node_id += 1

# Sort nodes by time ascending
nodes.sort(key=lambda x: x['time'])

# Constants for arc generation
MinUtilTime = 400  # minimum cumulative arc travel time to generate

# Travel times and capacities for arcs (in minutes)
arc_data = {
    ("X", "X"): (0, 150),
    ("X", "HSK"): (10, 150),
    ("X", "ATB"): (40, 150),
    ("HSK", "HSK"): (200, 150),
    ("ATB", "ATB"): (200, 150),
    ("HSK", "ATB"): (110, 1),
    ("ATB", "HSK"): (110, 1),
    ("HSK", "X"): (10, 150),
    ("ATB", "X"): (40, 150)
}

# Allowed alternation for chaining arcs
transitions = {
    "ATB": "HSK",
    "HSK": "ATB"
}

# Extract travel times for alternating arcs
travel_time = {
    ("ATB", "HSK"): arc_data[("ATB", "HSK")][0],
    ("HSK", "ATB"): arc_data[("HSK", "ATB")][0]
}

synthetic_id = len(nodes) + 1  # next id for synthetic destination nodes
E = []  # list to store arcs

# Generate arcs chains starting from each ATB/HSK node
for node in nodes:
    loc = node['loc']
    time = node['time']
    src_id = node['id']

    # Only start chains from ATB or HSK nodes
    if loc not in transitions:
        continue

    total_time = 0
    curr_loc = loc
    curr_time = time
    curr_src_id = src_id

    while total_time < MinUtilTime:
        next_loc = transitions[curr_loc]
        cost = travel_time[(curr_loc, next_loc)]
        next_time = curr_time + cost

        if next_time > 1440:  # don't exceed the day end
            break

        # Create synthetic destination node with new ID, time, and location
        dst = {'id': synthetic_id, 'time': next_time, 'loc': next_loc}
        synthetic_id += 1

        # Append arc dict with src, dst, cost, and capacity
        E.append({
            'src': {'id': curr_src_id, 'time': curr_time, 'loc': curr_loc},
            'dst': dst,
            'cost': cost,
            'cap': 1
        })

        # Update for next iteration (continue chaining)
        total_time += cost
        curr_loc = next_loc
        curr_time = next_time
        curr_src_id = dst['id']
        
    cost_to_X, cap_to_X = arc_data[(curr_loc, 'X')]
    E.append({
        'src': {'id': curr_src_id, 'time': curr_time, 'loc': curr_loc},
        'dst': {'id': synthetic_id, 'time': curr_time + cost_to_X, 'loc': 'X'},
        'cost': cost_to_X,
        'cap': cap_to_X
    })
    synthetic_id += 1




# Print count and sample arcs
print(f"Total generated arcs: {len(E)}")
for arc in E[:10]:
    print(f"From node {arc['src']['id']} ({arc['src']['loc']} @ {arc['src']['time']}) "
          f"to node {arc['dst']['id']} ({arc['dst']['loc']} @ {arc['dst']['time']}) "
          f"cost: {arc['cost']}")


Total generated arcs: 126
From node 2 (ATB @ 1020) to node 45 (HSK @ 1130) cost: 110
From node 45 (HSK @ 1130) to node 46 (ATB @ 1240) cost: 110
From node 46 (ATB @ 1240) to node 47 (HSK @ 1350) cost: 110
From node 47 (HSK @ 1350) to node 48 (X @ 1360) cost: 10
From node 5 (HSK @ 1020) to node 49 (ATB @ 1130) cost: 110
From node 49 (ATB @ 1130) to node 50 (HSK @ 1240) cost: 110
From node 50 (HSK @ 1240) to node 51 (ATB @ 1350) cost: 110
From node 51 (ATB @ 1350) to node 52 (X @ 1390) cost: 40
From node 3 (ATB @ 1040) to node 53 (HSK @ 1150) cost: 110
From node 53 (HSK @ 1150) to node 54 (ATB @ 1260) cost: 110


In [61]:
from datetime import timedelta

# Frequencies of nodes per hour at ATB and HSK
freq1 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 5, 6, 2, 0, 0]  # ATB
freq2 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 5, 6, 4, 0, 0]  # HSK

# freq1 = [0, 0, 0, 0, 3, 3, 4, 8, 10, 10, 12, 16, 9, 15, 15, 16, 14, 12, 9, 6, 6, 4, 0, 0]
# freq2 = [0, 0, 0, 0, 4, 6 ,5, 8, 10, 10, 12, 16, 9, 15, 15, 16, 14, 12, 9, 6, 6, 4, 0, 0]

# Generate nodes with ids, times, and locations
nodes = []
node_id = 1

# Add start node at 00:00 (location 'X')
nodes.append({'id': node_id, 'time': 0, 'loc': 'X'})
start_node_id = node_id
node_id += 1

for hour in range(24):
    f1 = freq1[hour]
    f2 = freq2[hour]

    # Generate ATB nodes for this hour
    for i in range(f1):
        time_val = hour * 60 + (i * 60 // max(f1, 1))
        nodes.append({'id': node_id, 'time': time_val, 'loc': 'ATB'})
        node_id += 1

    # Generate HSK nodes for this hour
    for i in range(f2):
        time_val = hour * 60 + (i * 60 // max(f2, 1))
        nodes.append({'id': node_id, 'time': time_val, 'loc': 'HSK'})
        node_id += 1

# Add end node at 24:00 (location 'X')
nodes.append({'id': node_id, 'time': 1440, 'loc': 'X'})
node_id += 1

# Sort nodes by time ascending
nodes.sort(key=lambda x: x['time'])

# Constants for arc generation
MinUtilTime = 400

# Travel times and capacities for arcs (in minutes)
arc_data = {
    ("X", "X"): (0, 150),
    ("X", "HSK"): (10, 150),
    ("X", "ATB"): (40, 150),
    ("HSK", "HSK"): (200, 150),
    ("ATB", "ATB"): (200, 150),
    ("HSK", "ATB"): (110, 1),
    ("ATB", "HSK"): (110, 1),
    ("HSK", "X"): (10, 150),
    ("ATB", "X"): (40, 150)
}

transitions = {
    "ATB": "HSK",
    "HSK": "ATB"
}

travel_time = {
    ("ATB", "HSK"): arc_data[("ATB", "HSK")][0],
    ("HSK", "ATB"): arc_data[("HSK", "ATB")][0]
}

synthetic_id = len(nodes) + 1
E = []

# Arcs from 'X' to all ATB/HSK nodes (if reachable)
for node in nodes:
    loc = node['loc']
    dst_time = node['time']

    if loc not in ['ATB', 'HSK']:
        continue

    cost = arc_data[('X', loc)][0]
    cap = arc_data[('X', loc)][1]
    T=dst_time-cost

    if dst_time >= cost:
        E.append({
            'src': {'id': start_node_id, 'time': T, 'loc': 'X'},
            'dst': {'id': node['id'], 'time': dst_time, 'loc': loc},
            'cost': cost,
            'cap': cap
        })

# Chained arcs
for node in nodes:
    loc = node['loc']
    time = node['time']
    src_id = node['id']

    if loc not in transitions:
        continue

    total_time = 0
    curr_loc = loc
    curr_time = time
    curr_src_id = src_id

    while total_time < MinUtilTime:
        next_loc = transitions[curr_loc]
        cost = travel_time[(curr_loc, next_loc)]
        next_time = curr_time + cost

        if next_time > 1440:
            break

        dst = {'id': synthetic_id, 'time': next_time, 'loc': next_loc}
        synthetic_id += 1

        E.append({
            'src': {'id': curr_src_id, 'time': curr_time, 'loc': curr_loc},
            'dst': dst,
            'cost': cost,
            'cap': 1
        })

        total_time += cost
        curr_loc = next_loc
        curr_time = next_time
        curr_src_id = dst['id']

    cost_to_X, cap_to_X = arc_data[(curr_loc, 'X')]
    E.append({
        'src': {'id': curr_src_id, 'time': curr_time, 'loc': curr_loc},
        'dst': {'id': synthetic_id, 'time': curr_time + cost_to_X, 'loc': 'X'},
        'cost': cost_to_X,
        'cap': cap_to_X
    })
    synthetic_id += 1

for node in nodes:
    node['hhmm'] = str(timedelta(minutes=node['time']))[:-3]

# print(E)
count=0
# Output preview
for node in nodes:
    # if(node['loc']=='HSK'):
        print(node)
        count+=1
print(count)

# Print count and sample arcs
print(f"Total generated arcs: {len(E)}")
for arc in E[:]:
    print(f"From node {arc['src']['id']} ({arc['src']['loc']} @ {arc['src']['time']}) "
          f"to node {arc['dst']['id']} ({arc['dst']['loc']} @ {arc['dst']['time']}) "
          f"cost: {arc['cost']}")

{'id': 1, 'time': 0, 'loc': 'X', 'hhmm': '0:00'}
{'id': 2, 'time': 1020, 'loc': 'ATB', 'hhmm': '17:00'}
{'id': 5, 'time': 1020, 'loc': 'HSK', 'hhmm': '17:00'}
{'id': 3, 'time': 1040, 'loc': 'ATB', 'hhmm': '17:20'}
{'id': 6, 'time': 1040, 'loc': 'HSK', 'hhmm': '17:20'}
{'id': 4, 'time': 1060, 'loc': 'ATB', 'hhmm': '17:40'}
{'id': 7, 'time': 1060, 'loc': 'HSK', 'hhmm': '17:40'}
{'id': 8, 'time': 1080, 'loc': 'ATB', 'hhmm': '18:00'}
{'id': 12, 'time': 1080, 'loc': 'HSK', 'hhmm': '18:00'}
{'id': 9, 'time': 1095, 'loc': 'ATB', 'hhmm': '18:15'}
{'id': 13, 'time': 1095, 'loc': 'HSK', 'hhmm': '18:15'}
{'id': 10, 'time': 1110, 'loc': 'ATB', 'hhmm': '18:30'}
{'id': 14, 'time': 1110, 'loc': 'HSK', 'hhmm': '18:30'}
{'id': 11, 'time': 1125, 'loc': 'ATB', 'hhmm': '18:45'}
{'id': 15, 'time': 1125, 'loc': 'HSK', 'hhmm': '18:45'}
{'id': 16, 'time': 1140, 'loc': 'ATB', 'hhmm': '19:00'}
{'id': 21, 'time': 1140, 'loc': 'HSK', 'hhmm': '19:00'}
{'id': 17, 'time': 1152, 'loc': 'ATB', 'hhmm': '19:12'}
{'id': 